In [2]:
import os
import pandas as pd
from tqdm import tqdm
import json

def load_and_concatenate_jsons(directory):
    all_dfs = []
    files = sorted(f for f in os.listdir(directory) if f.startswith("train_") and f.endswith("rollouts.json"))[::3]
    print(f"Found {len(files)} JSON files in {directory}")
    for fname in tqdm(files, desc="Loading JSON files", total=len(files)):
        try:
            iter_num = int(fname.split("_")[1])  # Extract iteration number from filename, e.g., train_1_rollouts.json → 1
            path = os.path.join(directory, fname)
            with open(path, 'r') as f:
                data = json.load(f)
            df = pd.DataFrame(data)
            df["iter"] = iter_num
            all_dfs.append(df)
        except Exception as e:
            print(f"Error loading {fname}: {e}")

    return pd.concat(all_dfs, ignore_index=True)



def load_file(filename):
    try:
        iter_num = int(os.path.basename(filename).split("_")[1])  # Extract iteration number from filename, e.g., train_1_rollouts.json → 1
        with open(filename, 'r') as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        df["iter"] = iter_num
        # all_dfs.append(df)
    except Exception as e:
        print(f"Error loading {filename}: {e}")
    return df


# def load_jsonl_stream(path):
#     with open(path, 'r') as f:
#         for line in f:
#             yield json.loads(line)

# def load_and_concatenate_jsonl(directory):
#     all_rows = []
#     files = sorted(f for f in os.listdir(directory) if f.startswith("train_") and f.endswith("rollouts.json"))

#     for fname in tqdm(files, desc="Loading JSONL files"):
#         try:
#             iter_num = int(fname.split("_")[1])
#             path = os.path.join(directory, fname)
#             for item in load_jsonl_stream(path):
#                 item["iter"] = iter_num
#                 all_rows.append(item)
#         except Exception as e:
#             print(f"Error loading {fname}: {e}")

#     return pd.DataFrame(all_rows)


# Usage
# directory = "/project/flame/asetlur/checkpoints/math-curriculum/Math/16klen-qwen3easy8koldckpt-hard64-b64mb32n16-crh0.35l0.2_redlog"
# directory = "/project/flame/asetlur/checkpoints/math-curriculum/Math/16klen-qwen3easy8koldckpt-hard2500-b64mb32n16micb16-crh0.35l0.2_redlog"
directory = "/project/flame/asetlur/checkpoints/math-curriculum/Math/16klen-qwen3easy8koldckpt-medium2500-b320mb160n16genmb64-crh0.35l0.2"
# dfs = load_and_concatenate_jsons(directory)
df = load_file(os.path.join(directory, "train_53_rollouts.json"))
# directory = "/project/flame/asetlur/checkpoints/math-curriculum/Math/16klen-qwen3easy8koldckpt-hard2500-b128mb32n16micb32-crh0.35l0.2_redlog"
# directory = "/project/flame/asetlur/checkpoints/math-curriculum/Math/16klen-qwen3easy8koldckpt-hard2500-b320mb160n16-crh0.35l0.2_redlog"
# directory = "/project/flame/asetlur/checkpoints/math-curriculum/Math/16klen-qwen3easy8koldckpt-hard2500-b320mb32n16micb32-crh0.35l0.2-fp32_redlog"
# dfl = load_and_concatenate_jsons(directory)

In [3]:
import numpy as np
df['indices'] = df['indices'].astype(np.int32)
df.columns

Index(['responses', 'response_mask', 'indices', 'input_texts', 'output_texts',
       'score', 'entropy', 'ratio', 'iter'],
      dtype='object')

In [4]:
df['indices'].head()

0     4123
1    10097
2     2677
3     3861
4     2897
Name: indices, dtype: int32

In [11]:
np.array(dfs[dfs['indices']==2461]['response_mask'].tolist()).sum(), np.array(dfl[dfl['indices']==2461]['response_mask'].tolist()).sum() 
# dfl[dfl['indices']==83]['response_mask'].sum()

(np.int64(122953), np.int64(213402))

In [13]:
dfs[dfs['indices']==2461]['score'].mean(), dfl[dfl['indices']==2461]['score'].mean()

(np.float64(0.9375), np.float64(0.25))

In [ ]:
print(df.columns)
import numpy as np
ent = np.array(df['entropy'].tolist())

mask = np.array(df['response_mask'].tolist())
print(
    ent.shape, 
    mask.shape, 
    (ent * mask).sum() / mask.sum(), 
    mask.sum(), 
    mask.sum() / (mask.shape[0] * mask.shape[1]),
    df['score'].mean())

In [ ]:
print(df.columns)
import numpy as np
ent = np.array(df['entropy'].tolist())

mask = np.array(df['response_mask'].tolist())
print(
    ent.shape, 
    mask.shape, 
    (ent * mask).sum() / mask.sum(), 
    mask.sum(), 
    mask.sum() / (mask.shape[0] * mask.shape[1]),
    df['score'].mean())

In [ ]:
(ent * mask).sum() / mask.sum()

In [ ]:
(ent * mask).sum() / mask.sum()

In [ ]:
(ent * mask).sum() / mask.sum()

In [ ]:
mask.mean()

In [ ]:
df["indices"] = df["indices"].astype(int)
iters = sorted(df["iter"].unique().tolist())
print(f"Total iterations: {len(iters)} Iterations: {iters}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Sort iterations
iters = sorted(df["iter"].unique())  # Ensure 'iters' is defined
cmap = cm.get_cmap("viridis", len(iters))  # Choose a colormap

fig, ax = plt.subplots(figsize=(8, 5))  # Use explicit Axes

index = df["indices"].unique()[62]

for i, iter in enumerate(iters):
    entropy = np.array(df[(df["indices"] == index) & (df["iter"] == iter)]["entropy"].tolist())
    response_mask = np.array(df[(df["indices"] == index) & (df["iter"] == iter)]["response_mask"].tolist())
    masked_entropy = entropy[response_mask == 1]
    values = masked_entropy
    thresholds = np.linspace(np.min(values), np.max(values), 200)
    frac_greater = [(values > a).mean() for a in thresholds]
    
    color = cmap(i)
    ax.plot(thresholds, frac_greater, color=color, label=f"Iter {iter}", alpha=0.8)
    
    # Plot vertical dashed line at the mean
    mean_val = np.mean(values)
    ax.axvline(mean_val, color=color, linestyle="--", linewidth=1.2, alpha=0.8)

# Colorbar setup
norm = mcolors.Normalize(vmin=min(iters), vmax=max(iters))
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Required for older matplotlib versions
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Iteration")

# Plot formatting
ax.set_xlabel("Threshold a")
ax.set_ylabel("Fraction of values > a")
ax.set_title("Complementary CDF (1 - CDF) across Iterations")
ax.grid(True)
plt.tight_layout()
plt.ylim(0, 0.6)
plt.xlim(0, 1.5)
plt.show()
